In [6]:
import pandas as pd
import numpy as np
import joblib

In [7]:
validation_set = pd.read_csv("validation.csv")
validation_set

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,2025-11-01,0.90402,1.86388
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,2025-11-01,0.92851,1.86240
2,TE-000003,San Francisco,Raleigh,35.19670,-121.69849,35.37376,-77.47605,2846.6,Dry Van,33435.0,2025-11-01,0.88117,1.95480
3,TE-000004,Dayton,Los Angeles,39.87206,-85.62931,28.56624,-116.70249,2261.6,Dry Van,25392.0,2025-11-01,0.89156,2.14579
4,TE-000005,Memphis,San Antonio,35.56802,-89.52871,29.50969,-98.40059,800.4,Flatbed,NaN,2025-11-01,0.88308,2.22541
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,TE-011996,Fort Wayne,Louisville,41.31561,-85.36206,37.36588,-87.47153,377.4,Dry Van,38638.0,2025-12-31,1.03391,2.15984
11996,TE-011997,Oklahoma City,Memphis,31.88294,-98.58097,35.56802,-89.52871,707.4,Dry Van,27685.0,2025-12-31,0.99846,2.46279
11997,TE-011998,Nashville,Montgomery,35.29479,-88.08915,31.37602,-86.10743,359.4,Flatbed,35522.0,2025-12-31,0.98564,2.09568
11998,TE-011999,Syracuse,San Antonio,42.80915,-76.89973,29.50969,-98.40059,1750.8,Dry Van,31755.0,2025-12-31,1.03100,1.95125


In [8]:
final_model = joblib.load("freight_rate_model.pkl")
final_kmeans = joblib.load("geographic_kmeans.pkl")

# Load validation data
validation = pd.read_csv("validation.csv")

In [9]:
validation["date"] = pd.to_datetime(validation["date"])

validation["dayofweek"] = validation["date"].dt.dayofweek
validation["month"] = validation["date"].dt.month

validation["is_weekend"] = (
    validation["dayofweek"].isin([5, 6]).astype(int)
)

validation["is_month_end"] = (
    validation["date"].dt.is_month_end.astype(int)
)

validation["dow_sin"] = np.sin(
    2 * np.pi * validation["dayofweek"] / 7
)

validation["dow_cos"] = np.cos(
    2 * np.pi * validation["dayofweek"] / 7
)

validation["month_sin"] = np.sin(
    2 * np.pi * validation["month"] / 12
)

validation["month_cos"] = np.cos(
    2 * np.pi * validation["month"] / 12
)

In [10]:
def add_regions(df, kmeans_model):

    df = df.copy()

    pickup = df[
        ["pickup_lat", "pickup_lon"]
    ].copy()

    pickup.columns = ["lat", "lon"]

    delivery = df[
        ["delivery_lat", "delivery_lon"]
    ].copy()

    delivery.columns = ["lat", "lon"]

    df["pickup_region"] = kmeans_model.predict(pickup)
    df["delivery_region"] = kmeans_model.predict(delivery)

    return df

In [11]:
validation = add_regions(
    validation,
    final_kmeans
)

In [12]:
features = [
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "equipment",
    "pickup_region",
    "delivery_region",
    "is_weekend",
    "is_month_end",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos"
]

In [13]:
X_validation = validation[features]

validation["predicted_rate"] = final_model.predict(
    X_validation
).round(2)

In [14]:
validation.head(5)

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,...,month,is_weekend,is_month_end,dow_sin,dow_cos,month_sin,month_cos,pickup_region,delivery_region,predicted_rate
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,...,11,1,0,-0.974928,-0.222521,-0.5,0.866025,6,4,794.87
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,...,11,1,0,-0.974928,-0.222521,-0.5,0.866025,8,1,4786.78
2,TE-000003,San Francisco,Raleigh,35.19670,-121.69849,35.37376,-77.47605,2846.6,Dry Van,33435.0,...,11,1,0,-0.974928,-0.222521,-0.5,0.866025,9,8,5000.69
3,TE-000004,Dayton,Los Angeles,39.87206,-85.62931,28.56624,-116.70249,2261.6,Dry Van,25392.0,...,11,1,0,-0.974928,-0.222521,-0.5,0.866025,7,1,4002.50
4,TE-000005,Memphis,San Antonio,35.56802,-89.52871,29.50969,-98.40059,800.4,Flatbed,NaN,...,11,1,0,-0.974928,-0.222521,-0.5,0.866025,0,3,1721.13


In [15]:
submission = validation[
    ["load_id", "predicted_rate"]
].copy()

submission.to_csv(
    "validation-predictions-template.csv",
    index=False
)